In [1]:
from pathlib import Path

import numpy as np

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.ls_opt import logging as ls_opt_logging
from cardiac_electrophysiology.ls_opt import optimizer
from cardiac_electrophysiology.utils import analysis, visualization

In [2]:
posterior_settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_map_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.05,
        tau=10,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=100,
        noise_variance=1e-3,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

optimizer_settings = optimizer.LBFGSConfig(
    maximum_num_iterations=1000,
    relative_function_tolerance= 1e-6,
    relative_gradient_tolerance=1e-6,
    max_line_search_steps=100,
)
ls_opt_logger_settings = ls_opt_logging.LSOPTLoggerSettings(
    print_to_console=True,
    logfile_path= Path("../results/lsopt_logfile.log"),
)

In [3]:
posterior_builder = builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
visualization.visualize_data_points(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)

Widget(value='<iframe src="http://localhost:46603/index.html?ui=P_0x7fc337f13a10_0&reconnect=auto" class="pyvi…

In [4]:
initial_guess = additional_output.prior_mean_parameter
ls_optimizer = optimizer.LBFGSOptimizer(optimizer_settings, ls_opt_logger_settings)
map_result = ls_optimizer.run(
    initial_guess=initial_guess,
    loss_function=posterior.evaluate_cost,
    gradient_function=posterior.evaluate_gradient,
)
print(f"MAP estimation success: {map_result.success}")
print(f"Status message: {map_result.status_message}")
np.save("../results/map_estimate.npy", map_result.result)
np.save("../results/map_loss_history.npy", map_result.loss_history)
np.save("../results/map_gradient_norm_history.npy", map_result.gradient_norm_history)

| Iteration   | Time        | Loss        | Grad Norm   | 
---------------------------------------------------------
| +1.000e+00  | +8.516e+00  | +3.751e+04  | +1.184e+04  | 
| +2.000e+00  | +1.038e+01  | +3.531e+04  | +3.928e+04  | 
| +3.000e+00  | +1.130e+01  | +3.175e+04  | +1.722e+04  | 
| +4.000e+00  | +1.223e+01  | +2.993e+04  | +1.202e+04  | 
| +5.000e+00  | +1.320e+01  | +2.777e+04  | +1.845e+04  | 
| +6.000e+00  | +1.412e+01  | +2.636e+04  | +2.517e+04  | 
| +7.000e+00  | +1.504e+01  | +2.556e+04  | +1.743e+04  | 
| +8.000e+00  | +1.593e+01  | +2.513e+04  | +1.297e+04  | 
| +9.000e+00  | +1.687e+01  | +2.467e+04  | +1.464e+04  | 
| +1.000e+01  | +1.783e+01  | +2.409e+04  | +1.258e+04  | 
| +1.100e+01  | +1.869e+01  | +2.341e+04  | +1.452e+04  | 
| +1.200e+01  | +1.956e+01  | +2.218e+04  | +1.208e+04  | 
| +1.300e+01  | +2.040e+01  | +2.097e+04  | +1.891e+04  | 
| +1.400e+01  | +2.127e+01  | +2.029e+04  | +1.827e+04  | 
| +1.500e+01  | +2.218e+01  | +1.982e+04  | +9.075e+03  |

In [5]:
map_parameter = np.load("../results/map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

Prior mean angle L2-error: 47.6143838338968
Prior mean angle max-error: 1.0181236206278492
MAP angle L2-error: 36.30430261953175
MAP angle max-error: 0.9950155147523658
Prior mean predictive L2-error: 124.95818328857422
Prior mean predictive max-error: 4.491857528686523
MAP predictive L2-error: 32.66608428955078
MAP predictive max-error: 1.6333694458007812
Data predictive L2-error: 0.3073747858116663
Data predictive max-error: 0.11342273965972538


In [6]:
for data in (
    additional_output.prior_mean_parameter,
    additional_output.ground_truth_parameter,
    map_parameter,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=False,
        clim=(-0.9, 0.9),
    )
for data in (
    analysis_data.diff_lat_truth_prior,
    analysis_data.diff_lat_truth_map,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=False,
        clim=(-2, 4.5)
    )

Widget(value='<iframe src="http://localhost:46603/index.html?ui=P_0x7fc1f0c93ed0_1&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46603/index.html?ui=P_0x7fc1f12f1a90_2&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46603/index.html?ui=P_0x7fc1f12f3390_3&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46603/index.html?ui=P_0x7fc1f12f3c50_4&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:46603/index.html?ui=P_0x7fc1e8634550_5&reconnect=auto" class="pyvi…